In [3]:
#importing the libraries

import os
import chromadb

from openai import OpenAI
from dotenv import load_dotenv

In [4]:
#Loading the API Key

load_dotenv()

client=OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [5]:
#Preparing our documents

documents = [
    "Python is a popular programming language.",
    "Machine learning allows computers to learn from data.",
    "RAG combines document retrieval with language generation.",
    "Neural networks are inspired by the human brain.",
    "Embeddings represent text as numerical vectors.",
    "Vector databases store embeddings for similarity search."
]

In [6]:
#Creating IDs and Metadata
ids = [
    "doc1",
    "doc2",
    "doc3",
    "doc4",
    "doc5",
    "doc6"
]

metadatas = [
    {"topic": "Python", "source": "learning_notes"},
    {"topic": "Machine Learning", "source": "learning_notes"},
    {"topic": "RAG", "source": "learning_notes"},
    {"topic": "Neural Networks", "source": "learning_notes"},
    {"topic": "Embeddings", "source": "learning_notes"},
    {"topic": "Vector Databases", "source": "learning_notes"}
]

In [7]:
#Generating Embeddings

def get_embedding(text):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

In [8]:
embeddings = []

for document in documents:

    embedding = get_embedding(document)

    embeddings.append(embedding)

print("Number of embeddings:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

Number of embeddings: 6
Vector dimensions: 1536


In [9]:
#Connect to your persistent ChromaDB database

chroma_client=chromadb.PersistentClient(
    path="./chroma_db"
)

collection=chroma_client.get_or_create_collection(
    name="ai_learning_collection"
)

#Storing documents in ChromaDB
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print("Total Documents:",collection.count())

Total Documents: 6


In [10]:
#Retrieval function to retriev relevant chunks
def retrieve_documents(question, number_of_results=3):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )

    query_embedding = response.data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=number_of_results
    )

    return results

In [11]:
#Testing 
question = "What is RAG?"
results = retrieve_documents(question)

print(results["documents"])

[['RAG combines document retrieval with language generation.', 'Machine learning allows computers to learn from data.', 'Vector databases store embeddings for similarity search.']]


In [12]:
#Building the context as the LLM needs retrieved text in a readable format.

def build_context(results):

    documents = results["documents"][0]

    context = "\n\n".join(documents)

    return context

In [13]:
#Testing the context
context = build_context(results)

print(context)

RAG combines document retrieval with language generation.

Machine learning allows computers to learn from data.

Vector databases store embeddings for similarity search.


In [15]:
#generating an answer usnig LLM
def generate_answer(question, context):

    response = client.responses.create(
        model="gpt-4.1-mini",

        instructions="""
        You are a helpful AI assistant.

        Answer the user's question using only
        the provided context.

        If the answer is not available in the context,
        clearly say that you do not have enough information.

        Do not invent facts.
        """,

        input=f"""
        Context:
        {context}

        Question:
        {question}

        Answer:
        """
    )

    return response.output_text

In [16]:
#Combining everything into one
def rag_answer(question):

    results = retrieve_documents(question)

    context = build_context(results)

    answer = generate_answer(question, context)

    return answer

In [17]:
question = "What is RAG?"

answer = rag_answer(question)

print("Question:", question)

print("Answer:", answer)

Question: What is RAG?
Answer: RAG combines document retrieval with language generation.


In [18]:
#Adding source information
def rag_answer_with_sources(question):

    results = retrieve_documents(question)

    context = build_context(results)

    answer = generate_answer(question, context)

    sources = results["metadatas"][0]

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }

In [19]:
#Testing information
result = rag_answer_with_sources(
    "What is RAG?"
)

print("Answer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(source)

Answer:
RAG combines document retrieval with language generation.

Sources:
{'topic': 'RAG', 'source': 'learning_notes'}
{'source': 'learning_notes', 'topic': 'Machine Learning'}
{'source': 'learning_notes', 'topic': 'Vector Databases'}


In [21]:
#checking with unknown information to check whether it's inventing things
question = "Who invented Python?"

answer = rag_answer(question)

print(answer)

I do not have enough information to answer who invented Python.
